# Multi-Factor Quant MVP — Interactive Exploration

本 notebook 演示如何在 Jupyter 中交互式地调用框架各模块：
1. 加载数据
2. 计算 MOM_6M 因子
3. 预处理
4. IC 分析
5. 五分位回测
6. 可视化

In [ ]:
import sys
from pathlib import Path

# 把工程根目录加入 sys.path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import build_wide_tables, get_universe
from src.factors import get_factor
from src.preprocessing import preprocess_factor
from src.analysis import compute_ic, ic_summary, ic_summary_table
from src.backtest import quintile_backtest
from src.visualization.plots_mpl import plot_ic_series_mpl, plot_quintile_nav_mpl

## 1. 获取股票池 & 宽表

In [ ]:
uni = get_universe()
print('S&P 500 tickers:', len(uni))
uni.head()

In [ ]:
# 快速测试可只取前 50 只股票
wide = build_wide_tables(tickers=uni['ticker'].head(50).tolist())
adj_close = wide['adj_close']
returns = wide['returns']
print('adj_close shape:', adj_close.shape)
adj_close.tail()

## 2. 计算 MOM_6M 因子

In [ ]:
factor = get_factor('MOM_6M')
raw = factor.compute(adj_close)
print('raw shape:', raw.shape)
raw.tail()

## 3. 预处理 (MAD 去极值 + Z-score)

In [ ]:
clean = preprocess_factor(raw)
clean.describe().T.head()

## 4. IC 分析

In [ ]:
ic = compute_ic(clean, returns)
print(ic_summary(ic))
fig = plot_ic_series_mpl(ic, title='MOM_6M IC Series')
fig

## 5. 五分位回测

In [ ]:
result = quintile_backtest(clean, returns, factor_direction=factor.direction)
print(result.config)
result.group_metrics

In [ ]:
fig = plot_quintile_nav_mpl(result.group_nav, result.long_short_nav, title='MOM_6M Quintile NAV')
fig

## 6. 多因子 IC 汇总（未来扩展示例）

当你在 `configs/default.yaml` 的 `factors.enabled` 中启用多个因子后，可用 `ic_summary_table` 生成与用户参考图一致的汇总表。

In [ ]:
# 示例：同时计算 MOM_6M 和 MOM_3M
from src.factors import FACTOR_REGISTRY
ic_dict = {}
for fname in ['MOM_6M', 'MOM_3M']:
    f = FACTOR_REGISTRY[fname]()
    x = preprocess_factor(f.compute(adj_close))
    ic_dict[fname] = compute_ic(x, returns)
ic_summary_table(ic_dict)